# STA 554 HW 10
Jillian Greene

# Part 1

In [1]:
# Import packages and initialize spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sqrt

spark = SparkSession.builder \
    .appName("StructuredStreamingRateExample") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 10:52:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Create streaing data by rate
rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

In [3]:
# Compute sq root of value & mod 4 of the rate
transformed_df = rate_df.select(col("timestamp"),
    col("value"),
    sqrt(col("value")).alias("sqrt_value"),
    (col("value") % 4).alias("mod_4"))

In [4]:
# Let this run for ~30 seconds before running next chunk
query = transformed_df.writeStream \
    .format("memory") \
    .queryName("rate_table") \
    .outputMode("append") \
    .start()

26/04/20 10:52:29 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-82abb2cd-5ba3-497c-92ca-8ca27feca066. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 10:52:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [5]:
# Stop query and show the accumulated output
query.stop()

spark.sql("SELECT * FROM rate_table").show()

26/04/20 10:53:00 WARN DAGScheduler: Failed to cancel job group e9f2b88c-3cdc-4985-83ac-ab955efe12bd. Cannot find active jobs for it.
26/04/20 10:53:00 WARN DAGScheduler: Failed to cancel job group e9f2b88c-3cdc-4985-83ac-ab955efe12bd. Cannot find active jobs for it.


+--------------------+-----+------------------+-----+
|           timestamp|value|        sqrt_value|mod_4|
+--------------------+-----+------------------+-----+
|2026-04-20 10:52:...|    0|               0.0|    0|
|2026-04-20 10:52:...|    1|               1.0|    1|
|2026-04-20 10:52:...|    2|1.4142135623730951|    2|
|2026-04-20 10:52:...|    3|1.7320508075688772|    3|
|2026-04-20 10:52:...|    4|               2.0|    0|
|2026-04-20 10:52:...|    5|  2.23606797749979|    1|
|2026-04-20 10:52:...|    6| 2.449489742783178|    2|
|2026-04-20 10:52:...|    7|2.6457513110645907|    3|
|2026-04-20 10:52:...|    8|2.8284271247461903|    0|
|2026-04-20 10:52:...|    9|               3.0|    1|
|2026-04-20 10:52:...|   10|3.1622776601683795|    2|
|2026-04-20 10:52:...|   11|   3.3166247903554|    3|
|2026-04-20 10:52:...|   12|3.4641016151377544|    0|
|2026-04-20 10:52:...|   13| 3.605551275463989|    1|
|2026-04-20 10:52:...|   14|3.7416573867739413|    2|
|2026-04-20 10:52:...|   15|

# Part 2

In [7]:
# Import packages and initialize new spark session for reproducability (in case I want to use this example in the future)
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("BikePipelineStreaming") \
    .getOrCreate()

# Read training data
bike_df = spark.read.csv("bikeDetails_for_fit.csv", header=True, inferSchema=True)

In [8]:
# Create transformer
sql_transformer = SQLTransformer(statement= """
    SELECT 
        log(selling_price) as label,
        year,
        log(km_driven) as log_km_driven,
        CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
    FROM __THIS__
    """)

# Assembler
assembler = VectorAssembler(inputCols=["year", "log_km_driven", "one_owner"], outputCol="features")

# Pipeline
pipeline = Pipeline(stages=[sql_transformer, assembler])

In [9]:
# Fit the model!
pipeline_model = pipeline.fit(bike_df)

To run this as per the HW 10 instructions, empty the `bikeStreaming` folder, run the following code chunk, then add the bikeDetails_add*.csv files one at a time. Then run the stop query when done

In [10]:
# Set up the data streaming process
# Schema
schema = bike_df.schema

# Streaming df
streaming_df = spark.readStream \
    .schema(schema) \
    .option("header", True) \
    .csv("bikeStreaming")

# Transform new data 
transformed_stream = pipeline_model.transform(streaming_df)

# Run query
query = transformed_stream.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

26/04/20 11:22:09 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-022a6ea6-3bda-4d25-b114-a83824c9618c. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 11:22:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

In [11]:
# Stop when done
query.stop()

26/04/20 11:22:50 WARN DAGScheduler: Failed to cancel job group d55891e6-102e-4092-8cda-93394f41fdb4. Cannot find active jobs for it.
26/04/20 11:22:50 WARN DAGScheduler: Failed to cancel job group d55891e6-102e-4092-8cda-93394f41fdb4. Cannot find active jobs for it.
